# Davion AI — Multi-Dataset Brain Tumor Retraining (Colab T4 GPU)

Notebook ini mengunduh beberapa dataset MRI tumor otak global (**Figshare via Hugging Face**, **Kaggle Brain Tumor MRI**, dan **Br35H**), menyatukannya lewat `ml.datasets.harmonizer` (zero patient leakage), melatih `DavionTumorEnsemble` di GPU T4, dan mengekspor `davion_ensemble_best.pt` + `metrics.json` untuk dipasang di aplikasi web.

In [ ]:
!git clone https://github.com/davinmaritza/davion-ai.git || (cd davion-ai && git pull)
%cd /content/davion-ai
!pip install -q -r ml/requirements.txt datasets kagglehub

In [ ]:
# 1. Unduh Figshare Brain Tumor Dataset (3,064 slices, patient-disjoint) dari Hugging Face (tanpa API key)
from pathlib import Path
from datasets import load_dataset

figshare_root = Path('/content/datasets/figshare')
ds = load_dataset('MedOtter/Figshare_Brain_Tumor')

for split_name, split_ds in ds.items():
    label_feat = split_ds.features.get('label')
    for idx, row in enumerate(split_ds):
        raw_label = row['label']
        if hasattr(label_feat, 'int2str') and isinstance(raw_label, int):
            cls_name = str(label_feat.int2str(raw_label)).lower().replace(' ', '_')
        else:
            cls_name = str(raw_label).lower().replace(' ', '_')
        out_dir = figshare_root / cls_name
        out_dir.mkdir(parents=True, exist_ok=True)
        pid = row.get('patient_id', row.get('PID', f'{split_name}_{idx}'))
        img = row['image']
        img.save(out_dir / f'PID-{pid}_{split_name}_{idx}.png')

print('Figshare exported to:', figshare_root)

In [ ]:
# 2. Unduh dataset Kaggle (Masoud Nickparvar 4-class + Br35H) via kagglehub
import os, shutil
import kagglehub

datasets_found = ['figshare=/content/datasets/figshare']

kaggle_targets = {
    'kaggle_mri': 'masoudnickparvar/brain-tumor-mri-dataset',
    'br35h': 'ahmedhamada0/brain-tumor-detection',
}

for name, slug in kaggle_targets.items():
    try:
        path = kagglehub.dataset_download(slug)
        datasets_found.append(f'{name}={path}')
        print(f'Downloaded {name}: {path}')
    except Exception as exc:
        print(f'Skipped {name} ({slug}): {exc}')

os.environ['DAVION_DATASETS'] = ' '.join(datasets_found)
print('Active datasets:', os.environ['DAVION_DATASETS'])

In [ ]:
# 3. Jalankan Training Multi-Dataset di Tesla T4 GPU
!python -m ml.train \
  --datasets $DAVION_DATASETS \
  --output-dir artifacts/checkpoints \
  --epochs 15 \
  --batch-size 16 \
  --grad-accum 2

In [ ]:
# 4. Evaluasi Holdout & Ekspor metrics.json + Download Checkpoint untuk Web App
from pathlib import Path
from ml.evaluate import evaluate_checkpoint

metrics = evaluate_checkpoint(
    checkpoint_path=Path('artifacts/checkpoints/davion_ensemble_best.pt'),
    manifest_csv=Path('artifacts/checkpoints/manifest.csv'),
    output_json=Path('artifacts/checkpoints/metrics.json'),
)
print('Overall Accuracy:', metrics['overall_accuracy'])
print('Macro ROC-AUC:', metrics['macro_roc_auc'])
print('Calibration ECE:', metrics['ece'])
print('Per Dataset:', metrics['per_dataset'])

try:
    from google.colab import files
    files.download('artifacts/checkpoints/davion_ensemble_best.pt')
    files.download('artifacts/checkpoints/metrics.json')
except ImportError:
    pass